In [ ]:
import pandas as pd
df = pd.read_csv("data/processed/player_match_stats.csv")
print(df.shape)
df.head()


In [ ]:
avg_bat_position = df.groupby("player")["batting_position"].agg(lambda x: x.mode()[0])

In [ ]:
print(avg_bat_position)

In [ ]:
def bin_batting_position(pos):
    if pos == 0:
        return 0  # didn't bat
    elif pos <= 3:
        return 1  # top order
    elif pos <= 6:
        return 2  # middle order
    else:
        return 3  # lower order

df["batting_position_bucket"] = df["batting_position"].apply(bin_batting_position)

In [ ]:
df["batting_position_bucket"].isna()

In [ ]:
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values(["player","date"]).reset_index(drop = True)
df[["player","date","total_fantasy_points"]].head(20)

In [ ]:
df["Total_career_runs"] = df.groupby(["player"])["runs"].transform(lambda x : x.shift(1).expanding().sum())
df["Total_career_wickets"] = df.groupby(["player"])["wickets"].transform(lambda x : x.shift(1).expanding().sum())


In [ ]:
df[df["player"] == "V Kohli"][["player", "date", "runs", "wickets","Total_career_runs" , "Total_career_wickets"]].head(10)
df[df["player"] == "JJ Bumrah"][["player", "date", "runs", "wickets", "Total_career_runs", "Total_career_wickets"]]


In [ ]:
df[["player","Total_career_runs" , "Total_career_wickets"]]

In [ ]:
df["rolling_avg_fantasy_5"] = df.groupby("player")["total_fantasy_points"].transform(lambda x: x.shift(1).rolling(5, min_periods = 1).mean())
df["rolling_std_fantasy_5"] = df.groupby("player")["total_fantasy_points"].transform(lambda x: x.shift(1).rolling(5, min_periods = 1).std())
df[["player", "date", "total_fantasy_points", "rolling_avg_fantasy_5","rolling_std_fantasy_5"]].head(20)

In [ ]:
df[df["player"] == "V Kohli"][["player", "date", "total_fantasy_points", "rolling_avg_fantasy_5","rolling_std_fantasy_5"]]

In [ ]:
df["rolling_avg_fantasy_3"] = df.groupby("player")["total_fantasy_points"].transform(lambda x: x.shift(1).rolling(3,min_periods= 1).mean())

df["rolling_std_fantasy_10"] = df.groupby("player")["total_fantasy_points"].transform(lambda x: x.shift(1).rolling(5,min_periods = 1).std())

df["rolling_avg_fantasy_10"] = df.groupby("player")["total_fantasy_points"].transform(lambda x: x.shift(1).rolling(10,min_periods= 1).mean())

df["rolling_avg_runs_5"] = df.groupby("player")["runs"].transform(lambda x: x.shift(1).rolling(5,min_periods = 1).mean())

df["rolling_avg_wickets_5"] = df.groupby("player")["wickets"].transform(lambda x: x.shift(1).rolling(5,min_periods = 1).mean())

df["matches_played"] = df.groupby("player")["date"].transform(lambda x: x.expanding().count().shift(1).fillna(0))


print("Form features added")
df[["player", "date", "total_fantasy_points", "rolling_avg_fantasy_3", 
    "rolling_avg_fantasy_5", "rolling_avg_fantasy_10", 
    "rolling_avg_runs_5", "rolling_avg_wickets_5", 
    "matches_played","rolling_std_fantasy_10"]].head(20)

In [ ]:
df["expanding_season_fantasy_avg"] = df.groupby(["player" , "season"])["total_fantasy_points"].transform( lambda x: x.shift(1).expanding().mean())
df["expanding_season_fantasy_std"] = df.groupby(["player", "season"])["total_fantasy_points"].transform(lambda x : x.shift(1).expanding().std())

In [ ]:
df["batting_contribution"] = df.apply(lambda row: row["runs"]/row["team_total"] if row["team_total"] > 0 else 0, axis = 1)
df["bowling_contribution"] = df.apply(lambda row: row["wickets"]/row["total_wickets"] if row["total_wickets"] > 0 else 0, axis = 1)
print(df[["player", "runs", "team_total", "batting_contribution", "wickets", "total_wickets", "bowling_contribution"]].head(10))

In [ ]:
print(df[df["player"] == "BB McCullum"][["player", "runs", "team_total", "batting_contribution"]].head(3))

In [ ]:
df["rolling_batting_contribution_5"] = (
    df.groupby("player")["batting_contribution"].transform(lambda x: x.shift(1).rolling(5, min_periods = 1).mean())
)
df["rolling_bowling_contribution_5"] = (
    df.groupby("player")["bowling_contribution"].transform(lambda x: x.shift(1).rolling(5,min_periods = 1).mean())
)


In [ ]:
print(df[df["player"] == "BB McCullum"][["player", "runs", "team_total", "rolling_batting_contribution_5"]].head(10))

In [ ]:
df[df["player"] == "V Kohli"][["player", "date", "total_fantasy_points", "rolling_avg_fantasy_3", 
    "rolling_avg_fantasy_5", "rolling_avg_fantasy_10", 
    "rolling_avg_runs_5", "rolling_avg_wickets_5", 
    "matches_played","rolling_std_fantasy_5","rolling_std_fantasy_10","expanding_season_fantasy_avg","expanding_season_fantasy_std"]].head(30)

In [ ]:
HOME_CITIES = { 
    "Mumbai Indians": "Mumbai",
    "Chennai Super Kings": "Chennai",
    "Royal Challengers Bangalore": "Bengaluru",
    "Royal Challengers Bengaluru": "Bengaluru",
    "Kolkata Knight Riders": "Kolkata",
    "Sunrisers Hyderabad": "Hyderabad",
    "Rajasthan Royals": "Jaipur",
    "Punjab Kings": "Mohali",
    "Kings XI Punjab": "Mohali",
    "Delhi Capitals": "Delhi",
    "Delhi Daredevils": "Delhi",
    "Gujarat Titans": "Ahmedabad",
    "Lucknow Super Giants": "Lucknow",
    "Deccan Chargers": "Hyderabad",
    "Kochi Tuskers Kerala": "Kochi",
    "Pune Warriors": "Pune",
    "Rising Pune Supergiants": "Pune"
}

In [ ]:
df["is_home"] = (df["team"].map(HOME_CITIES) == df["city"]).astype(int)

In [ ]:
df[["team", "city", "is_home"]]


In [ ]:
def venue_avg_before(group):
    return group.shift(1).expanding().mean()

def venue_std_before(group):
    return group.shift(1).expanding().std()


df["venue_avg_fantasy"] = (
    df.groupby(["player","venue"])["total_fantasy_points"]
    .transform(venue_avg_before)
    )

df["venue_std_fantasy"] = (
    df.groupby(["player","venue"])["total_fantasy_points"]
    .transform(venue_std_before)
)



df[["player", "date", "venue", "total_fantasy_points", "venue_avg_fantasy", "venue_std_fantasy"]].head(30)

In [ ]:
career_stats = df.groupby("player").agg(
    total_matches = ("player", "count"),
    career_runs = ("runs", "sum"),
    career_balls_bowled = ("balls_bowled","sum"),
    career_wickets = ("wickets", "sum"),
    avg_bat_position = ("batting_position", lambda x: x.mode()[0])
).reset_index()

print(career_stats[career_stats["player"].isin(["V Kohli", "JJ Bumrah", "HH Pandya"])])

In [ ]:
def classify_role(total_matches, career_runs, career_wickets, avg_bat_position,career_balls_bowled):
    is_genuine_bowler = career_wickets >= 20 and career_balls_bowled >= 200
    is_genuine_batter = is_genuine_batter = career_runs >= 500 or (career_runs >= 300 and avg_bat_position <= 7)

    if total_matches > 20:
        if is_genuine_batter and is_genuine_bowler:
            role = "allrounder"
        elif is_genuine_bowler:
            role = "bowler"
        elif is_genuine_batter:
            role = "batter"
        else:
            role = "unknown"
    else:
        if avg_bat_position == 0:
            role = "bowler"
        elif career_balls_bowled > 50 and avg_bat_position >= 7:
            role = "bowler"
        elif avg_bat_position <= 7 and career_balls_bowled > 30:
            role = "allrounder"
        elif avg_bat_position <= 7:
            role = "batter"
        else:
            role = "bowler"
        
    return role

In [ ]:
career_stats["role"] = career_stats.apply(
    lambda row: classify_role(
        row["total_matches"],
        row["career_runs"],
        row["career_wickets"],
        row["avg_bat_position"],
        row["career_balls_bowled"]
    ),
    axis=1
)

In [ ]:

print(career_stats[career_stats["player"].isin(["V Kohli", "JJ Bumrah", "HH Pandya"])])

In [ ]:
print(career_stats["role"].value_counts())

In [ ]:
df = df.drop(columns=["role"], errors="ignore")
df = df.merge(career_stats[["player", "role"]], on="player", how="left")

In [ ]:
print(df[["player", "role"]].head(10))
print(df["role"].value_counts())

In [ ]:
df["role_encoded"] = df["role"].map({
    "batter": 0,
    "bowler": 1,
    "allrounder": 2,
    "unknown": 3
})

In [ ]:
def oppositon_avg_before(group):
    return group.shift(1).expanding().mean()

def oppositon_std_before(group):
    return group.shift(1).expanding().std()

df["opposition_avg_fantasy"] = (
    df.groupby(["player", "opposition"])["total_fantasy_points"]
    .transform(oppositon_avg_before)
    )

df["opposition_std_fantasy"] = (
    df.groupby(["player", "opposition"])["total_fantasy_points"]
    .transform(oppositon_std_before)
    )

print("Opposition feature added")
print(df.shape)
print(df.isnull().sum())

In [ ]:
match_innings = df[df["player_innings"] != 0].drop_duplicates(subset = ["match_id","player_innings"])[
       ["match_id", "venue", "date", "player_innings", "team_total"]  
    ]

In [ ]:
match_pivot = match_innings.pivot(index = "match_id", columns= "player_innings",values= "team_total")
match_pivot.columns = ["innings1_score", "innings2_score"]
match_pivot = match_pivot.reset_index()

print(match_pivot.head(10))

In [ ]:
match_meta = df.drop_duplicates(subset="match_id")[["match_id","venue","date"]]
match_level = match_pivot.merge(match_meta, on = "match_id", how = "left")
match_level["date"] = pd.to_datetime(match_level["date"])
match_level = match_level.sort_values("date").reset_index(drop = True)

print(match_level.head(10))
print(match_level.shape)

In [ ]:
match_level["venue_avg_innings1"] = (
    match_level.groupby("venue")["innings1_score"]
    .transform(lambda x : x.shift(1).expanding().mean())
)

match_level["venue_avg_innings2"] = (
    match_level.groupby("venue")["innings2_score"]
    .transform(lambda x : x.shift(1).expanding().mean())
)

match_level["venue_avg_total_runs"] = (
    match_level.groupby("venue").apply(
        lambda g : (g["innings1_score"] + g["innings2_score"]).shift(1).expanding().mean()
    ).reset_index(level = 0, drop = True)
)

print(match_level[["venue", "date", "innings1_score", "innings2_score", "venue_avg_innings1", "venue_avg_innings2"]].head(20))

In [ ]:
venue_features = match_level[["match_id", "venue_avg_innings1", "venue_avg_innings2", "venue_avg_total_runs"]]
df = df.merge(venue_features, on = "match_id", how = "left")

print(df.shape)
print(df[["player", "match_id", "venue", "venue_avg_innings1", "venue_avg_innings2"]].head(10))

In [ ]:
unique_cities = df["city"].dropna().unique()
print(len(unique_cities))
print(unique_cities)

In [ ]:
import requests

CITY_NAME_OVERRIDES = {
    "Bangalore": "Bengaluru",
    "New Chandigarh": "Mullanpur",
}


def geocode_city(city_name, country_code = None):
    city_name = CITY_NAME_OVERRIDES.get(city_name,city_name)
    url = "https://geocoding-api.open-meteo.com/v1/search"
    params = {"name": city_name, "count": 5}
    response = requests.get(url, params = params, timeout= 10)
    if response.status_code == 200:
        data = response.json()
        if "results" in data:
            for result in data["results"]:
                if country_code is None or result.get("country_code") == country_code:
                    return result["latitude"], result["longitude"]
    return None , None
    

In [ ]:
print(geocode_city("Bangalore", country_code="IN"))
print(geocode_city("New Chandigarh", country_code="IN"))
print(geocode_city("Mumbai", country_code="IN"))

In [ ]:
import time

INDIAN_CITIES = {
    "Bangalore", "Delhi", "Chandigarh", "Mumbai", "Kolkata", "Jaipur", "Hyderabad",
    "Chennai", "Ahmedabad", "Cuttack", "Nagpur", "Dharamsala", "Kochi", "Indore",
    "Visakhapatnam", "Pune", "Raipur", "Ranchi", "Rajkot", "Kanpur", "Bengaluru",
    "Navi Mumbai", "Lucknow", "Guwahati", "Mohali", "New Chandigarh"
}

city_coords = {}

for city in unique_cities:
    if city == "Unknown":
        continue
    country_code = "IN" if city in INDIAN_CITIES else None
    lat , lon = geocode_city(city,country_code = country_code)
    city_coords[city] = {"lat": lat , "lon": lon}
    print(city, lat, lon)
    time.sleep(0.2)

In [ ]:
city_date_ranges = df.groupby("city")["date"].agg(["min","max"])
print(city_date_ranges)

In [ ]:
def fetch_city_weather_range(lat,lon,start_date,end_date):
    url = "https://archive-api.open-meteo.com/v1/archive"
    params = {
        "latitude": lat,
        "longitude": lon,
        "start_date":start_date,
        "end_date": end_date,
        "hourly": "temperature_2m,relative_humidity_2m,dew_point_2m,wind_speed_10m,precipitation",
        "timezone": "auto"
    }
    response = requests.get(url,params = params, timeout = 30)
    if response.status_code == 200:
        return response.json()
    else:
        print("Failed:", response.status_code, response.text[:200])
        return None

In [ ]:
result = fetch_city_weather_range(19.07283, 72.88261, "2008-04-20", "2026-05-24")
print(result["hourly"].keys())
print(len(result["hourly"]["time"]))

In [ ]:
import pandas as pd

def extract_match_day_weather(weather_json, match_date):
    times = weather_json["hourly"]["time"]
    temps = weather_json["hourly"]["temperature_2m"]
    humidity = weather_json["hourly"]["relative_humidity_2m"]
    dew = weather_json["hourly"]["dew_point_2m"]
    wind = weather_json["hourly"]["wind_speed_10m"]
    precip = weather_json["hourly"]["precipitation"]

    match_temps, match_humidity, match_dew, match_wind, match_precip = [],[],[],[],[]

    for i , t in enumerate(times):
        if t.startswith(match_date):
            hour = int(t.split("T")[1].split(":")[0])
            if 15 <= hour <= 21:
                match_temps.append(temps[i])
                match_humidity.append(humidity[i])
                match_dew.append(dew[i])
                match_wind.append(wind[i])
                match_precip.append(precip[i])
    if not  match_temps:
        return None
 
    return {
        "temp": sum(match_temps) / len(match_temps),
        "humidity": sum(match_humidity)/ len(match_humidity),
        "dew": sum(match_dew) / len(match_dew),
        "windspeed": sum(match_wind) / len(match_wind),
        "precip": sum(match_precip)
    }

result_day = extract_match_day_weather(result, "2024-04-09")
print(result_day)

In [806]:
import time as time_module

weather_lookup = {}
fetched_cities = set()

for city in unique_cities:
    if city == "Unknown" or city in fetched_cities:
        continue

    coords = city_coords.get(city)
    if coords is None or coords["lat"] is None:
        print(f"skipping {city},NO COORDINATES")
        continue
    city_matches = df[df["city"] == city]
    start_date = city_matches["date"].min().strftime("%Y-%m-%d")
    end_date = city_matches["date"].max().strftime("%Y-%m-%d")

    print(f"fetching{city}: {start_date} to {end_date}")
    weather_json = fetch_city_weather_range(coords["lat"], coords["lon"],start_date, end_date)

    if weather_json is None:
        print(f"Failed to fetch {city}, will retry later")
        time_module.sleep(15)  # back off longer after a failure
        weather_json = fetch_city_weather_range(coords["lat"], coords["lon"], start_date, end_date)
        if weather_json is None:
            print(f"Skipping {city} after retry failure")
            continue

    
    unique_dates_for_city = city_matches["date"].dt.strftime("%Y-%m-%d").unique()

    for match_date in unique_dates_for_city:
        day_weather = extract_match_day_weather(weather_json, match_date)
        weather_lookup[f"{city},{match_date}"] = day_weather
    
    fetched_cities.add(city)
    time_module.sleep(0.5)

print(f"Done. Total weather entries: {len(weather_lookup)}, cities fetched: {len(fetched_cities)}")
    

fetchingBangalore: 2008-04-18 to 2017-05-19
7 7 7 7 7
7 7 7 7 7
7 7 7 7 7
7 7 7 7 7
7 7 7 7 7
7 7 7 7 7
7 7 7 7 7
7 7 7 7 7
7 7 7 7 7
7 7 7 7 7
7 7 7 7 7
7 7 7 7 7
7 7 7 7 7
7 7 7 7 7
7 7 7 7 7
7 7 7 7 7
7 7 7 7 7
7 7 7 7 7
7 7 7 7 7
7 7 7 7 7
7 7 7 7 7
7 7 7 7 7
7 7 7 7 7
7 7 7 7 7
7 7 7 7 7
7 7 7 7 7
7 7 7 7 7
7 7 7 7 7
7 7 7 7 7
7 7 7 7 7
7 7 7 7 7
7 7 7 7 7
7 7 7 7 7
7 7 7 7 7
7 7 7 7 7
7 7 7 7 7
7 7 7 7 7
7 7 7 7 7
7 7 7 7 7
7 7 7 7 7
7 7 7 7 7
7 7 7 7 7
7 7 7 7 7
7 7 7 7 7
7 7 7 7 7
7 7 7 7 7
7 7 7 7 7
7 7 7 7 7
7 7 7 7 7
7 7 7 7 7
7 7 7 7 7
7 7 7 7 7
7 7 7 7 7
7 7 7 7 7
7 7 7 7 7
7 7 7 7 7
7 7 7 7 7
7 7 7 7 7
7 7 7 7 7
7 7 7 7 7
7 7 7 7 7
7 7 7 7 7
7 7 7 7 7
7 7 7 7 7
7 7 7 7 7
fetchingDelhi: 2008-04-19 to 2026-05-17
7 7 7 7 7
7 7 7 7 7
7 7 7 7 7
7 7 7 7 7
7 7 7 7 7
7 7 7 7 7
7 7 7 7 7
7 7 7 7 7
7 7 7 7 7
7 7 7 7 7
7 7 7 7 7
7 7 7 7 7
7 7 7 7 7
7 7 7 7 7
7 7 7 7 7
7 7 7 7 7
7 7 7 7 7
7 7 7 7 7
7 7 7 7 7
7 7 7 7 7
7 7 7 7 7
7 7 7 7 7
7 7 7 7 7
7 7 7 7 7
7 7 7 7 7
7 7 7 7 7
7 7 7 

KeyboardInterrupt: 

In [ ]:
df.to_csv("data/processed/player_match_features.csv", index=False)
print("Saved")

In [ ]:
df["venue_avg_innings1"] = df["venue_avg_innings1"].fillna(df["venue_avg_innings1"].mean())
df["venue_avg_innings2"] = df["venue_avg_innings2"].fillna(df["venue_avg_innings2"].mean())
df["venue_avg_total_runs"] = df["venue_avg_total_runs"].fillna(df["venue_avg_total_runs"].mean())

In [ ]:
df["venue_first_appearance"] = df["venue_avg_fantasy"].isnull().astype(int)
df["opposition_first_appearance"] = df["opposition_avg_fantasy"].isnull().astype(int)

df["venue_avg_fantasy"] = df["venue_avg_fantasy"].fillna(0)
df["venue_std_fantasy"] = df["venue_std_fantasy"].fillna(0)

df["opposition_avg_fantasy"] = df["opposition_avg_fantasy"].fillna(0)
df["opposition_std_fantasy"] = df["opposition_std_fantasy"].fillna(0)

df["expanding_season_fantasy_avg"] = df["expanding_season_fantasy_avg"].fillna(0)
df["expanding_season_fantasy_std"] = df["expanding_season_fantasy_std"].fillna(0)

df["rolling_batting_contribution_5"] = df["rolling_batting_contribution_5"].fillna(0)
df["rolling_bowling_contribution_5"] = df["rolling_bowling_contribution_5"].fillna(0)

df["rolling_avg_fantasy_3"] = df["rolling_avg_fantasy_3"].fillna(0)
df["rolling_avg_fantasy_5"] = df["rolling_avg_fantasy_5"].fillna(0)
df["rolling_avg_fantasy_10"] = df["rolling_avg_fantasy_10"].fillna(0)
df["rolling_std_fantasy_5"] = df["rolling_std_fantasy_5"].fillna(0)
df["rolling_std_fantasy_10"] = df["rolling_std_fantasy_10"].fillna(0)
df["rolling_avg_runs_5"] = df["rolling_avg_runs_5"].fillna(0)
df["rolling_avg_wickets_5"] = df["rolling_avg_wickets_5"].fillna(0)
df["Total_career_runs"] = df["Total_career_runs"].fillna(0)
df["Total_career_wickets"] = df["Total_career_wickets"].fillna(0)


print(df.isnull().sum())

In [ ]:
import os
os.makedirs("data/processed", exist_ok= True)
df.to_csv("data/processed/player_match_features.csv", index = False)
print(len(df))
df["won_toss"] = (df["team"] == df["toss_winner"]).astype(int)
print(df["won_toss"])

In [ ]:
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date")

train = df[df["date"].dt.year < 2025]
test = df[df["date"].dt.year >=2025]

print(train.shape)
print(test.shape)
print(f"Train: {len(train)/len(df)*100:.1f}%")
print(f"Test: {len(test)/len(df)*100:.1f}%")

In [ ]:
print(len(train))
print(len(test))

In [ ]:
print(train.columns)

In [ ]:
df[["team", "toss_winner", "won_toss"]].head(20)

In [ ]:
df.columns

In [ ]:
df.shape

In [ ]:
features = [
"rolling_avg_fantasy_5",
"rolling_avg_fantasy_10",
"rolling_std_fantasy_10",
"rolling_avg_runs_5",
"rolling_avg_wickets_5",
"batting_position",
"matches_played",
"venue_std_fantasy",
"opposition_std_fantasy",
"venue_first_appearance",
"opposition_first_appearance",
"won_toss",
"expanding_season_fantasy_std",
"is_home",
"role_encoded",
"rolling_bowling_contribution_5",
"rolling_batting_contribution_5",
"venue_avg_innings1",
"venue_avg_innings2",
"venue_avg_total_runs"
]

target = "total_fantasy_points"

X_train = train[features]
y_train = train[target]
X_test = test[features]
y_test = test[target]

In [ ]:
print(X_train.shape)
print(y_train.shape)
print(X_test.shape)
print(y_test.shape)

In [ ]:
y_test_reset = y_test.reset_index(drop=True)
y_pred = X_test["rolling_avg_fantasy_5"].reset_index(drop=True)
error = abs(y_pred - y_test_reset)
MAE = error.mean()
print(MAE)

In [ ]:
from xgboost import XGBRegressor
regressor = XGBRegressor(
    n_estimators=500,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)
regressor.fit(X_train, y_train)
regressor.predict(X_test)

In [ ]:
import numpy as np
y_predi = regressor.predict(X_test)
error_X = abs(y_predi - y_test.values)
MAE = np.mean(error_X)
print(MAE)

In [ ]:
y_pred_train = regressor.predict(X_train)
error_train = abs(y_pred_train - y_train.values)
MAE_train = np.mean(error_train)
print(f"Train MAE: {MAE_train}")
print(f"Test MAE: {MAE}")

In [ ]:
importance = pd.Series(
    regressor.feature_importances_,
    index=features
).sort_values(ascending=False)
print(importance)

In [ ]:
X_train_single = train[["batting_position"]]
X_test_single = test[["batting_position"]]

from lightgbm import LGBMRegressor
lgbm_single = LGBMRegressor(n_estimators=500, max_depth=4, learning_rate=0.05, 
                              subsample=0.8, colsample_bytree=0.8, random_state=42)
lgbm_single.fit(X_train_single, y_train)

preds_single = lgbm_single.predict(X_test_single)
mae_single = np.mean(np.abs(preds_single - y_test.values))
print(f"MAE using ONLY batting_position: {mae_single:.2f}")

In [ ]:
from lightgbm import LGBMRegressor

lgbm = LGBMRegressor(
    n_estimators=500,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)
lgbm.fit(X_train, y_train)

In [ ]:
y_predi = lgbm.predict(X_test)
error_X = abs(y_predi - y_test.values)
MAE = np.mean(error_X)
print(MAE)

In [ ]:
y_pred_train = lgbm.predict(X_train)
MAE_train = np.mean(abs(y_pred_train - y_train.values))
print(f"Train MAE: {MAE_train}")
print(f"Test MAE: {MAE}")

In [ ]:
import pickle
import os

os.makedirs("models", exist_ok=True)

with open("models/lgbm_baseline.pkl", "wb") as f:
    pickle.dump(lgbm, f)

print("Model saved")

In [ ]:
os.makedirs("data/processed", exist_ok= True)
df.to_csv("data/processed/player_match_features.csv", index = False)
print(len(df))

In [ ]:
df.columns == "Nan"

In [ ]:
os.makedirs("data/processed", exist_ok= True)
df.to_csv("data/processed/player_match_features.csv", index = False)
print(len(df))

In [ ]:
unique_weather_queries = df[["date", "city"]].drop_duplicates()
print(len(unique_weather_queries))